In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting, datasets, image
from utils.get_aal import get_aal
from scipy.ndimage import center_of_mass
from matplotlib import pyplot as plt

npg_colors_18 = [
    '#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE',
    '#AA3377', '#BBBBBB', '#4477AA', '#EE6677', '#228833',
    '#CCBB44', '#66CCEE', '#AA3377', '#BBBBBB', '#4477AA',
    '#EE6677', '#228833', '#CCBB44'
]

# Get atlases
atlas_schaeffer = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7, resolution_mm=2)
atlas_img_schaeffer = image.load_img(atlas_schaeffer.maps)
atlas_schaeffer_coords = np.array([plotting.find_xyz_cut_coords(image.math_img(f"img=={i}", img=atlas_img_schaeffer)) for i in range(1, 401)])

atlas_aal = get_aal()
atlas_img_aal = image.load_img(atlas_aal.maps)
atlas_aal_coords = np.array([plotting.find_xyz_cut_coords(image.math_img(f"img=={atlas_aal.indices[i]}", img=atlas_img_aal)) for i in range(166)])

# Get ROI names
rois = ["rAmygdala_L", "rAmygdala_R", "rCaudate_L", "rCaudate_R", "rCerebellum_10_L", "rCerebellum_10_R", 
        "rCerebellum_3_L", "rCerebellum_3_R", "rCerebellum_4_5_L", "rCerebellum_4_5_R", "rCerebellum_6_L", 
        "rCerebellum_6_R", "rCerebellum_7b_L", "rCerebellum_7b_R", "rCerebellum_8_L", "rCerebellum_8_R", 
        "rCerebellum_9_L", "rCerebellum_9_R", "rCerebellum_Crus1_L", "rCerebellum_Crus1_R", "rCerebellum_Crus2_L", 
        "rCerebellum_Crus2_R", "rHippocampus_L", "rHippocampus_R", "rLC_L", "rLC_R", "rN_Acc_L", "rN_Acc_R", 
        "rPallidum_L", "rPallidum_R", "rParaHippocampal_L", "rParaHippocampal_R", "rPutamen_L", "rPutamen_R", 
        "rRaphe_D", "rRaphe_M", "rRed_N_L", "rRed_N_R", "rSN_pc_L", "rSN_pc_R", "rSN_pr_L", "rSN_pr_R", 
        "rThal_AV_L", "rThal_AV_R", "rThal_IL_L", "rThal_IL_R", "rThal_LGN_L", "rThal_LGN_R", "rThal_LP_L", 
        "rThal_LP_R", "rThal_MDl_L", "rThal_MDl_R", "rThal_MDm_L", "rThal_MDm_R", "rThal_MGN_L", "rThal_MGN_R", 
        "rThal_PuA_L", "rThal_PuA_R", "rThal_PuI_L", "rThal_PuI_R", "rThal_PuL_L", "rThal_PuL_R", "rThal_PuM_L", 
        "rThal_PuM_R", "rThal_Re_L", "rThal_Re_R", "rThal_VA_L", "rThal_VA_R", "rThal_VL_L", "rThal_VL_R", 
        "rThal_VPL_L", "rThal_VPL_R", "rVTA_L", "rVTA_R", "rVermis_10", "rVermis_3", "rVermis_4_5", "rVermis_6", 
        "rVermis_7", "rVermis_8", "rVermis_9"]

# Get ROI coordinates
idx = [np.where(np.array(atlas_aal.labels) == a.replace("r", "", 1))[0] for a in rois]
idx = [id[0] for id in idx if id.shape[0]>0]

roi_coords = np.concatenate([atlas_schaeffer_coords, atlas_aal_coords[idx, :]])

schaeffer_labels = np.array(atlas_schaeffer.labels, dtype=str)
aal_labels = np.array(atlas_aal.labels, dtype=str)[idx]
roi_names = np.concatenate([schaeffer_labels, aal_labels])

# Create network and get 4 most important regions
network = pd.read_csv("../external/schaeffer_subcortical_smith_templates.csv")


In [ ]:
network = pd.read_csv("../external/schaeffer_subcortical_smith_templates.csv")

In [ ]:
for i in range(18):
    net_name = network.iloc[i, 0]
    tmp = (network.iloc[i, 1:].to_numpy() - network.iloc[i, 1:].to_numpy().mean()) / network.iloc[i, 1:].to_numpy().std()
    display = plotting.plot_markers(tmp, roi_coords, node_size=tmp*18, node_threshold=2, node_cmap=plt.cm.gray, display_mode='z', annotate=False, colorbar=False)
    plt.savefig(f"../external/images/{net_name}.png", transparent=True)
    plt.close() 

print("Connectome plots saved to PNG files.")


In [ ]:
from nilearn import datasets
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=7, resolution_mm=1)
from utils.get_aal import get_aal
aal = get_aal()
from nilearn.image import new_img_like, load_img

# Save Schaefer atlas as a NIfTI file
schaefer_filename = '../../external/schaefer.nii.gz'

new_img_like(schaefer.maps, load_img(schaefer.maps).get_fdata()).to_filename(schaefer_filename)

# Save AAL atlas as a NIfTI file
aal_filename = '../../external/aal.nii.gz'
new_img_like(aal.maps, load_img(aal.maps).get_fdata()).to_filename(aal_filename)
import pandas as pd

# Create a TSV file for Schaefer atlas labels
schaefer_labels = pd.DataFrame({
    'region_id': range(1, 401),
    'region_name': schaefer.labels
})
schaefer_labels.to_csv('../../external/schaefer_labels.tsv', sep='\t', index=False)

# Create a TSV file for AAL atlas labels
aal_labels = pd.DataFrame({
    'region_id': range(1, len(aal.labels) + 1),
    'region_name': aal.labels
})
aal_labels.to_csv('../../external/aal_labels.tsv', sep='\t', index=False)
import json

# Create a JSON file for Schaefer atlas metadata
schaefer_metadata = {
    'atlas_name': 'Schaefer2018',
    'n_rois': 400,
    'yeo_networks': 7,
    'resolution_mm': 2,
    'description': 'Schaefer 2018 atlas with 400 regions and 7 Yeo networks at 2mm resolution.'
}
with open('../../external/schaefer_metadata.json', 'w') as f:
    json.dump(schaefer_metadata, f, indent=4)

# Create a JSON file for AAL atlas metadata
aal_metadata = {
    'atlas_name': 'AAL',
    'n_rois': len(aal.labels),
    'description': 'Automated Anatomical Labeling (AAL) atlas.'
}
with open('../../external/aal_metadata.json', 'w') as f:
    json.dump(aal_metadata, f, indent=4)